<a href="https://colab.research.google.com/github/MallikaJangam/BDA/blob/main/bda_assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Classification Model with Spark

In [3]:
!pip install pyspark

#  Start Spark
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Classification").getOrCreate()

# Load dataset
!wget -O /content/adult.csv https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race",
    "sex", "capital_gain", "capital_loss", "hours_per_week",
    "native_country", "income"
]

df = spark.read.csv("/content/adult.csv", header=False, inferSchema=True)
df = df.toDF(*columns)

# Select required columns
df = df.select("age", "education_num", "hours_per_week", "income")

# Clean income column (remove spaces issue)
from pyspark.sql.functions import trim
df = df.withColumn("income", trim(df["income"]))

#  Convert label (target variable)
from pyspark.sql.functions import when
df = df.withColumn("label", when(df["income"] == ">50K", 1).otherwise(0))

# Create feature vector
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["age", "education_num", "hours_per_week"],
    outputCol="features"
)

data = assembler.transform(df).select("features", "label")

#  Split data
train, test = data.randomSplit([0.8, 0.2], seed=42)

# Train model
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    regParam=0.1,  # Added regularization parameter (L2 here)
    elasticNetParam=0.0 # Set to 0.0 for L2 regularization
)
model = lr.fit(train)

# Predictions
predictions = model.transform(test)

# Show sample predictions, including probability and rawPrediction
predictions.select("features", "label", "prediction", "probability", "rawPrediction").show(5)

# Evaluate model accuracy and other metrics
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Accuracy
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy = accuracy_evaluator.evaluate(predictions)
print("Accuracy:", accuracy)

# F1-Score
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)
f1_score = f1_evaluator.evaluate(predictions)
print("F1 Score:", f1_score)

# Area Under ROC (AUC)
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = auc_evaluator.evaluate(predictions)
print("Area Under ROC (AUC):", auc)

# Area Under PR (AUPRC)
auprc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)
auprc = auprc_evaluator.evaluate(predictions)
print("Area Under PR (AUPRC):", auprc)

--2026-04-16 17:06:45--  https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘/content/adult.csv’

/content/adult.csv      [    <=>             ]   3.79M  3.56MB/s    in 1.1s    

2026-04-16 17:06:47 (3.56 MB/s) - ‘/content/adult.csv’ saved [3974305]

+---------------+-----+----------+--------------------+--------------------+
|       features|label|prediction|         probability|       rawPrediction|
+---------------+-----+----------+--------------------+--------------------+
|[17.0,4.0,40.0]|    0|       0.0|[0.95314636618547...|[3.01273990810596...|
|[17.0,5.0,15.0]|    0|       0.0|[0.96951973318918...|[3.45972134404560...|
|[17.0,5.0,16.0]|    0|       0.0|[0.96875435780705...|[3.43413116240622...|
|[17.0,5.0,20.0]|    0|  

Clustering Model with Spark

In [7]:
#Install PySpark
!pip install pyspark

#Start Spark
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Clustering").getOrCreate()

#Download dataset
!wget https://raw.githubusercontent.com/sharmaroshan/clustering-of-mall-customers/master/Mall_Customers.csv -O Mall_Customers.csv

#Load dataset
df = spark.read.csv("Mall_Customers.csv", header=True, inferSchema=True)

#Rename columns
df = df.withColumnRenamed("Annual Income (k$)", "income") \
       .withColumnRenamed("Spending Score (1-100)", "score")

#SELECT FEATURES (Better option)
# Try only income & score (often gives better clustering)
df = df.select("income", "score")

#Feature Vector
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["income", "score"],
    outputCol="features"
)

data = assembler.transform(df)

#Scaling
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withStd=True,
    withMean=True
)

data = scaler.fit(data).transform(data)

#Find BEST k automatically
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

best_k = 2
best_score = -1

print("Finding best k...\n")

for k in range(2, 11):
    kmeans = KMeans(k=k, featuresCol="scaledFeatures", seed=42)
    model = kmeans.fit(data)
    result = model.transform(data)

    evaluator = ClusteringEvaluator(featuresCol="scaledFeatures")
    score = evaluator.evaluate(result)

    print(f"k={k}, Silhouette Score={score}")

    if score > best_score:
        best_score = score
        best_k = k

print("\nBest k:", best_k)
print("Best Silhouette Score:", best_score)

#Final Model with best k
kmeans = KMeans(k=best_k, featuresCol="scaledFeatures", seed=42)
model = kmeans.fit(data)

result = model.transform(data)

#Show final clusters
result.select("income", "score", "prediction").show(10)

--2026-04-16 17:18:20--  https://raw.githubusercontent.com/sharmaroshan/clustering-of-mall-customers/master/Mall_Customers.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4286 (4.2K) [text/plain]
Saving to: ‘Mall_Customers.csv’

Mall_Customers.csv  100%[===================>]   4.19K  --.-KB/s    in 0s      

2026-04-16 17:18:20 (56.2 MB/s) - ‘Mall_Customers.csv’ saved [4286/4286]

Finding best k...

k=2, Silhouette Score=0.45270666773496565
k=3, Silhouette Score=0.6288672765684991
k=4, Silhouette Score=0.5107393443267989
k=5, Silhouette Score=0.7408351139612743
k=6, Silhouette Score=0.7249136389316958
k=7, Silhouette Score=0.7205063670066336
k=8, Silhouette Score=0.6898397738946709
k=9, Silhouette Score=0.6223941958736029
k=10, Silhouette Score=0.6661080

 Recommendation Engine with Spark

In [2]:
# Install PySpark
!pip install pyspark

#Start Spark Session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("RecommendationSystem").getOrCreate()
Download MovieLens Dataset (small)
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip

#Unzip dataset
!unzip -o ml-latest-small.zip

#Load ratings dataset
df = spark.read.csv("/content/ml-latest-small/ratings.csv", header=True, inferSchema=True)

# Show sample data
df.show(5)

# Select required columns
df = df.select("userId", "movieId", "rating")

#Train-Test Split
train, test = df.randomSplit([0.8, 0.2], seed=42)

# Build ALS Model
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",   # prevents NaN errors
    nonnegative=True,
    maxIter=10,
    regParam=0.1,
    rank=10
)

model = als.fit(train)

#Make Predictions
predictions = model.transform(test)

# Show predictions
predictions.select("userId", "movieId", "rating", "prediction").show(5)

#Evaluate Model (RMSE)
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)
print("RMSE:", rmse)

#Generate Top 5 Recommendations for each user
user_recommendations = model.recommendForAllUsers(5)

user_recommendations.show(truncate=False)

#(Optional): Recommendations for a specific user
user_id = 1
single_user = model.recommendForUserSubset(
    df.filter(df.userId == user_id), 5
)

single_user.show(truncate=False)

--2026-04-16 17:05:39--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  1.61MB/s    in 0.6s    

2026-04-16 17:05:40 (1.61 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982

In [5]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS

# Re-load ratings dataset for ALS to ensure correct 'train' and 'test' DataFrames
# Assuming ml-latest-small.zip has been downloaded and unzipped by a previous cell
df_als = spark.read.csv("/content/ml-latest-small/ratings.csv", header=True, inferSchema=True)
df_als = df_als.select("userId", "movieId", "rating")

# Re-split data for ALS model to get the correct 'train' and 'test'
train_als, test_als = df_als.randomSplit([0.8, 0.2], seed=42)

als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True
)

# Parameter tuning
paramGrid = ParamGridBuilder() \
    .addGrid(als.rank, [5, 10, 15]) \
    .addGrid(als.regParam, [0.01, 0.1, 0.2]) \
    .addGrid(als.maxIter, [5, 10]) \
    .build()

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

cv = CrossValidator(
    estimator=als,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3
)

# Use the correct train_als DataFrame
model = cv.fit(train_als)

best_model = model.bestModel
# Use the correct test_als DataFrame
predictions = best_model.transform(test_als)

rmse = evaluator.evaluate(predictions)
print("Best RMSE:", rmse)

Best RMSE: 0.87783083188673
